# Imports

In [0]:
from pyspark.sql import functions as F

# SETUP

In [0]:
%run ../config

# CONSTANTS

In [0]:
SOURCE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.iucr_codes"
TARGET_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.ref_iucr"

# INGESTION

In [0]:
df = spark.read.table(SOURCE_TABLE)
df.printSchema()

# TRANSFORM

In [0]:
df_silver = (
    df.select(
        "iucr",
        "primary_description",
        "secondary_description",
        "index_code",
        F.col("active").try_cast("boolean").alias("active"),
    )
    .withColumn("_silver_processed_at", F.current_timestamp())
)

# CHECK TH CAST

In [0]:
unreadable_active = df.filter(F.col("active").isNotNull() & F.col("active").try_cast("boolean").isNull()).count()
if unreadable_active:
    display(df.filter(F.col("active").isNotNull() & F.col("active").try_cast("boolean").isNull()).limit(10))
    raise ValueError(f"[IUCR] {unreadable_active:,} values of `active` cannot be cast to boolean (see the rows above)")

print("[OK] every value of `active` was cast")

# WRITE TO SILVER

In [0]:
(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

print(f"Loaded {spark.table(TARGET_TABLE).count()} rows into {TARGET_TABLE}")